# CS781 — continued pretraining of the Phase 1 language model (fixed recipe, held-out validation)

**Why this notebook exists.** The 3,000-step checkpoint was trained with an effective batch of 8 sequences (<= 32,768 tokens/step), a learning rate of 1e-5
(5e-5 in the Phase 4 config), and at most ~98M tokens in total, while the deduplicated corpus holds billions. The architecture is not the bottleneck; the training recipe and the token count are.

**What it does:** streams the deduplicated legal corpus from `cs781-phase4-training-prep` (never touching the validation/test manifests), packs it into 1024-token blocks with a domain-weighted mix
(more Supreme Court / tribunal text, like your downstream tasks), and trains with AdamW (beta2 0.95, weight decay 0.1 on matrices), warmup-stable-decay LR, 131k tokens per step, fp16/bf16, automatic resume.
Validation loss per domain on fixed held-out documents is logged against the 3,000-step baseline.

**Run order:** 1 setup -> 2 data -> 3 models -> 4 baselines (read-only) -> 5 trainer -> 6a smoke test -> 6b main run -> 7 export. (6c is an optional speed pilot.)

**Read before running**
* Check with the capstone spec whether a Phase 1 model continued after your submission may be used in Phase 3. This notebook does not decide that.
* GPU quota is shared with Phase 3 runs. Cell 6a prints how many steps fit in N GPU-hours; set `CFG['total_steps']` from it.
* The data-side logic (held-out exclusion, packing, domain mix, validation blocks, LR schedule) was tested on synthetic shards; the model/training cells need your GPU, so cell 6a is the first real test.


In [ ]:
# ================= CELL 1 — setup, paths, configuration =================
import os, sys, json, math, time, random, glob, gzip, gc, copy
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F

BASE   = "/kaggle/input/datasets/oipranesh/cs781-phase4-training-prep"          # dedup corpus + manifests + tokenizer
P1_CKPT = "/kaggle/input/datasets/oipranesh/cs781-phase-1-checkpoints/checkpoint_latest.pt"       # the 3,000-step model
P4_CKPT = "/kaggle/input/datasets/oipranesh/phase4-step1000-verified-model/checkpoint_final.pt"   # the Phase 4 step-1000 model
OUT = Path("/kaggle/working/pretrain"); OUT.mkdir(parents=True, exist_ok=True)
for p in (BASE, P1_CKPT): assert os.path.exists(p), f"missing: {p}"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert device.type == "cuda", "pretraining needs a GPU"
cap = torch.cuda.get_device_capability(0)
USE_BF16 = cap[0] >= 8                       # T4 (7.5) has no native bf16 -> fp16 + GradScaler there
print("GPU:", torch.cuda.get_device_name(0), "| capability", cap, "| precision:", "bf16" if USE_BF16 else "fp16 + GradScaler")

DOMAINS = ["supreme_court", "tribunals", "high_courts", "acts"]
CFG = dict(
    run="gpt2_warm_v1", arch="gpt2_warm",     # gpt2_warm = continue the 3,000-step weights | gpt2_scratch | llama_scratch (RoPE/RMSNorm/SwiGLU, same size)
    seq_len=1024, micro_bs=8, accum=16,       # 8*1024*16 = 131,072 tokens per optimizer step (the old runs used <= 32,768)
    lr_peak=3e-4, lr_min=3e-5, warmup=150, total_steps=2600, decay_start=None,    # warmup-stable-decay; decay_start defaults to 80% of total_steps
    wd=0.1, betas=(0.9, 0.95), clip=1.0, eval_every=200, eval_tokens_per_domain=262144,
    domain_w={"supreme_court": 0.30, "tribunals": 0.30, "high_courts": 0.35, "acts": 0.05},   # downstream texts are SC / tribunal judgments
    min_words=50, seed=1234)
print({k: v for k, v in CFG.items() if k != "domain_w"}, "\ndomain weights:", CFG["domain_w"])
print(f"tokens per step: {CFG['seq_len']*CFG['micro_bs']*CFG['accum']:,} | planned tokens: {CFG['seq_len']*CFG['micro_bs']*CFG['accum']*CFG['total_steps']/1e9:.2f}B")


In [ ]:
# ================= CELL 2 — streaming corpus, held-out split, packing, validation blocks =================
from tokenizers import Tokenizer
tok = Tokenizer.from_file(f"{BASE}/tokenizer/legal_bpe_32k.json")
EOS = tok.token_to_id("[EOS]"); assert EOS is not None, "tokenizer has no [EOS]"
V = tok.get_vocab_size(); print("tokenizer vocab:", V, "| EOS id:", EOS)

def shard_files(domain):
    fs = sorted(glob.glob(f"{BASE}/dedup/{domain}/*.jsonl") + glob.glob(f"{BASE}/dedup/{domain}/*.jsonl.gz")); assert fs, f"no shards for {domain}"; return fs
def open_text(p): return gzip.open(p, "rt", encoding="utf-8") if p.endswith(".gz") else open(p, encoding="utf-8")

def load_hash_set(split):
    s = set()
    with open(f"{BASE}/manifests/{split}_manifest.jsonl", encoding="utf-8") as f:
        for line in f: s.add(json.loads(line)["text_hash"])
    return s
HELD = {sp: load_hash_set(sp) for sp in ("validation", "test")}
SKIP = HELD["validation"] | HELD["test"]          # training never sees validation or test documents
print({k: f"{len(v):,} docs" for k, v in HELD.items()}, "| shards:", {d: len(shard_files(d)) for d in DOMAINS})

def iter_docs(domain, rng, skip, min_words, files=None):
    files = list(files or shard_files(domain))
    while True:
        rng.shuffle(files)
        for p in files:
            with open_text(p) as f:
                for line in f:
                    try: r = json.loads(line)
                    except Exception: continue
                    if r.get("text_hash") in skip: continue
                    t = r.get("text")
                    if not t: continue
                    try: wc = int(r.get("word_count"))
                    except Exception: wc = len(t.split())
                    if wc >= min_words: yield t

def shuffled(it, buf, rng):               # shard files are ordered (court/year), so mix them through a buffer
    pool = []
    for x in it:
        pool.append(x)
        if len(pool) >= buf:
            i = rng.randrange(len(pool)); pool[i], pool[-1] = pool[-1], pool[i]; yield pool.pop()

class PackedStream:
    """domain-weighted documents -> tokenised -> EOS-separated -> packed into [micro_bs, seq_len] blocks (no padding)"""
    def __init__(self, tokenizer, cfg, seed, skip):
        self.tok, self.cfg, self.rng = tokenizer, cfg, random.Random(seed); self.buf = []; self.dom_tokens = Counter()
        self.gens = {d: shuffled(iter_docs(d, random.Random(seed + i), skip, cfg["min_words"]), 1000, random.Random(seed + 100 + i)) for i, d in enumerate(DOMAINS)}
        self.w = [cfg["domain_w"][d] for d in DOMAINS]
    def _refill(self, need):
        while len(self.buf) < need:
            ds = [self.rng.choices(DOMAINS, weights=self.w)[0] for _ in range(32)]; texts = [next(self.gens[d]) for d in ds]
            for d, enc in zip(ds, self.tok.encode_batch(texts, add_special_tokens=False)):
                ids = enc.ids + [EOS]; self.buf.extend(ids); self.dom_tokens[d] += len(ids)
    def next_batch(self):
        B, T = self.cfg["micro_bs"], self.cfg["seq_len"]; self._refill(B * T)
        x = torch.tensor(self.buf[:B * T], dtype=torch.long).view(B, T); del self.buf[:B * T]; return x

def build_val_blocks(cfg, cache=OUT / "val_blocks.pt"):
    """fixed held-out blocks per domain, taken ONLY from validation-manifest documents, spread over several shards"""
    if cache.exists(): return torch.load(cache)
    T = cfg["seq_len"]; per = cfg["eval_tokens_per_domain"] // T; need = per * T; val = {}
    for d in DOMAINS:
        files = shard_files(d); idx = sorted({int(i * (len(files) - 1) / max(1, min(6, len(files)) - 1)) for i in range(min(6, len(files)))})
        quota = math.ceil(need / len(idx)); toks = []
        for fi in idx:
            got = 0
            with open_text(files[fi]) as f:
                for line in f:
                    try: r = json.loads(line)
                    except Exception: continue
                    if r.get("text_hash") in HELD["validation"] and r.get("text"):
                        ids = tok.encode(r["text"], add_special_tokens=False).ids + [EOS]; toks += ids; got += len(ids)
                        if got >= quota: break
        assert len(toks) >= need, f"{d}: only {len(toks)} validation tokens found (need {need})"
        val[d] = torch.tensor(toks[:need], dtype=torch.long).view(per, T)
    torch.save(val, cache); return val
VAL = build_val_blocks(CFG); print("validation blocks:", {d: tuple(v.shape) for d, v in VAL.items()}, "| tokens:", sum(v.numel() for v in VAL.values()))


In [ ]:
# ================= CELL 3 — models (same-size candidates) + validation loss =================
from transformers import GPT2Config, GPT2LMHeadModel, LlamaConfig, LlamaForCausalLM
def _make(cls, cfgobj):
    try: return cls._from_config(cfgobj, attn_implementation="sdpa")
    except Exception: return cls(cfgobj)

def build_model(arch):
    if arch.startswith("gpt2"):
        gc_ = GPT2Config(vocab_size=V, n_positions=4096, n_ctx=4096, n_embd=768, n_layer=12, n_head=12, resid_pdrop=0.0, embd_pdrop=0.0, attn_pdrop=0.0)
        m = _make(GPT2LMHeadModel, gc_)
        if arch == "gpt2_warm":
            ck = torch.load(P1_CKPT, map_location="cpu", weights_only=False); print("warm start:", m.load_state_dict(ck["model_state_dict"], strict=True)); del ck
        mc = dict(family="gpt2", vocab_size=V, context_length=4096, hidden_size=768, num_layers=12, num_heads=12)
    elif arch == "llama_scratch":
        lc = LlamaConfig(vocab_size=V, hidden_size=768, intermediate_size=2048, num_hidden_layers=12, num_attention_heads=12, num_key_value_heads=12,
                         max_position_embeddings=4096, rms_norm_eps=1e-5, rope_theta=10000.0, tie_word_embeddings=True, attention_bias=False, mlp_bias=False, initializer_range=0.02)
        m = _make(LlamaForCausalLM, lc)
        mc = dict(family="llama", vocab_size=V, context_length=4096, hidden_size=768, intermediate_size=2048, num_layers=12, num_heads=12)
    else: raise ValueError(arch)
    m.config.use_cache = False
    n = sum(p.numel() for p in m.parameters()); print(f"[{arch}] unique parameters: {n:,}")
    return m.to(device), mc

def amp_ctx(): return torch.autocast("cuda", dtype=torch.bfloat16 if USE_BF16 else torch.float16)

@torch.no_grad()
def evaluate(model, val, bs=8):
    model.eval(); res = {}
    for d, blocks in val.items():
        tot, n = 0.0, 0
        for i in range(0, len(blocks), bs):
            x = blocks[i:i + bs].to(device)
            with amp_ctx(): logits = model(input_ids=x).logits
            tot += F.cross_entropy(logits[:, :-1].float().reshape(-1, logits.shape[-1]), x[:, 1:].reshape(-1), reduction="sum").item(); n += x[:, 1:].numel()
        res[d] = tot / n
    res["overall"] = float(np.mean([res[d] for d in val])); model.train(); return res
def fmt(res): return " | ".join(f"{k} {v:.3f}" for k, v in res.items())


In [ ]:
# ================= CELL 4 — baselines: how good are the EXISTING checkpoints on held-out text? (read-only, ~3 min) =================
def load_into_gpt2(path):
    m, _ = build_model("gpt2_scratch"); ck = torch.load(path, map_location="cpu", weights_only=False, mmap=True)
    sd = ck.get("model_state_dict", ck)
    try: m.load_state_dict(sd, strict=True)
    except Exception as e: print("   could not load as GPT-2:", str(e)[:200]); return None, None
    return m, sd
BASELINE = {}
for name, path in (("phase1_step3000", P1_CKPT), ("phase4_step1000", P4_CKPT)):
    if not os.path.exists(path): print(name, "missing, skipped"); continue
    m, sd = load_into_gpt2(path)
    if m is None: continue
    BASELINE[name] = evaluate(m, VAL); print(f"{name:16s} val nats/token -> {fmt(BASELINE[name])}   (perplexity {math.exp(BASELINE[name]['overall']):.0f})")
    if name == "phase1_step3000": ref = {k: v.detach().cpu().float() for k, v in m.state_dict().items() if v.ndim >= 2}
    elif "ref" in globals():
        num = sum(((v.detach().cpu().float() - ref[k]) ** 2).sum().item() for k, v in m.state_dict().items() if k in ref)
        den = sum((v ** 2).sum().item() for v in ref.values())
        print(f"   relative distance between the 3,000-step and step-1000 weights: {math.sqrt(num/den):.3f}  "
              f"(small => same training lineage; ~1.4 => unrelated random initialisations)")
    del m, sd; gc.collect(); torch.cuda.empty_cache()
json.dump(BASELINE, open(OUT / "baseline.json", "w"))
print("\nuniform-guess loss would be", round(math.log(V), 2), "nats. Save these numbers: every later run is judged against them.")


In [ ]:
# ================= CELL 5 — training function (warmup-stable-decay, fp16/bf16, resume, validation, checkpoints) =================
def lr_at(step, c):
    w, T = c["warmup"], c["total_steps"]; ds = c["decay_start"] or int(0.8 * T)
    if step < w: return c["lr_peak"] * (step + 1) / w
    if step < ds: return c["lr_peak"]
    return c["lr_peak"] + (c["lr_min"] - c["lr_peak"]) * min(1.0, (step - ds) / max(1, T - ds))

def train(cfg, steps=None, accum=None, do_eval=True, save=True, resume=True, minutes_budget=None):
    c = {**cfg}; c["accum"] = accum or c["accum"]; rdir = OUT / c["run"]; rdir.mkdir(exist_ok=True)
    torch.manual_seed(c["seed"]); model, mc = build_model(c["arch"]); model.train()
    decay = [p for p in model.parameters() if p.ndim >= 2]; nodecay = [p for p in model.parameters() if p.ndim < 2]
    opt = torch.optim.AdamW([{"params": decay, "weight_decay": c["wd"]}, {"params": nodecay, "weight_decay": 0.0}], lr=c["lr_peak"], betas=c["betas"], eps=1e-8, fused=True)
    scaler = torch.amp.GradScaler("cuda", enabled=not USE_BF16); step, tokens = 0, 0
    ck_path = rdir / "latest.pt"
    if resume and ck_path.exists():
        ck = torch.load(ck_path, map_location="cpu", weights_only=False); model.load_state_dict(ck["model_state_dict"]); opt.load_state_dict(ck["optimizer_state_dict"])
        scaler.load_state_dict(ck["scaler_state_dict"]); step, tokens = ck["step"], ck["tokens"]; print(f"resumed from step {step}"); del ck
    stream = PackedStream(tok, c, c["seed"] + step, SKIP); T = c["seq_len"] * c["micro_bs"] * c["accum"]
    end = steps if steps is not None else c["total_steps"]; logf = open(rdir / "log.jsonl", "a"); t0 = time.time(); t_last, tok_last, last_save = time.time(), tokens, time.time()
    base = json.load(open(OUT / "baseline.json")) if (OUT / "baseline.json").exists() else {}; b_overall = base.get("phase1_step3000", {}).get("overall")
    best = float("inf"); run_loss, nb = 0.0, 0
    while step < end:
        lr = lr_at(step, c)
        for g in opt.param_groups: g["lr"] = lr
        for _ in range(c["accum"]):
            x = stream.next_batch().to(device)
            with amp_ctx(): loss = model(input_ids=x, labels=x).loss
            if not torch.isfinite(loss): print("non-finite loss, skipping microbatch"); continue
            scaler.scale(loss / c["accum"]).backward(); run_loss += loss.item(); nb += 1
        scaler.unscale_(opt); gn = torch.nn.utils.clip_grad_norm_(model.parameters(), c["clip"]); scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
        step += 1; tokens += T
        if step % 10 == 0:
            now = time.time(); tps = (tokens - tok_last) / (now - t_last); t_last, tok_last = now, tokens
            rec = dict(step=step, loss=run_loss / max(nb, 1), lr=lr, grad_norm=float(gn), tokens=tokens, tok_s=tps, elapsed_min=(now - t0) / 60); run_loss, nb = 0.0, 0
            logf.write(json.dumps(rec) + "\n"); logf.flush()
            if step % 50 == 0 or step <= 20: print(f"step {step:5d} | loss {rec['loss']:.3f} | lr {lr:.2e} | gnorm {rec['grad_norm']:.2f} | {tps/1e3:.1f}k tok/s | {rec['elapsed_min']:.1f} min | {tokens/1e6:.0f}M tokens", flush=True)
        if do_eval and (step % c["eval_every"] == 0 or step == end):
            r = evaluate(model, VAL); logf.write(json.dumps(dict(step=step, val=r)) + "\n"); logf.flush()
            gap = f" | vs 3000-step baseline {r['overall']-b_overall:+.3f}" if b_overall else ""
            print(f"   VALIDATION step {step}: {fmt(r)}{gap}", flush=True)
            if save and r["overall"] < best: best = r["overall"]; torch.save({"model_state_dict": model.state_dict(), "model_config": mc, "step": step, "val": r}, rdir / "best.pt")
        if save and time.time() - last_save > 20 * 60:
            torch.save({"model_state_dict": model.state_dict(), "optimizer_state_dict": opt.state_dict(), "scaler_state_dict": scaler.state_dict(), "step": step, "tokens": tokens, "model_config": mc, "cfg": c}, ck_path)
            last_save = time.time(); print("   checkpoint saved")
        if minutes_budget and (time.time() - t0) / 60 > minutes_budget: print(f"time budget reached at step {step}; saving and stopping"); break
    if save:
        torch.save({"model_state_dict": model.state_dict(), "optimizer_state_dict": opt.state_dict(), "scaler_state_dict": scaler.state_dict(), "step": step, "tokens": tokens, "model_config": mc, "cfg": c}, ck_path)
        torch.save({"model_state_dict": model.state_dict(), "model_config": {**mc, "trained_steps": step, "tokens": tokens}, "step": step}, rdir / "final_model.pt")
    logf.close(); print(f"done: step {step}, {tokens/1e6:.0f}M tokens, {(time.time()-t0)/60:.1f} min; domain token mix: " + str({d: f'{v/sum(stream.dom_tokens.values()):.2f}' for d, v in stream.dom_tokens.items()}))
    del opt, scaler; gc.collect(); torch.cuda.empty_cache(); return model

def plan_budget(tok_s, hours, cfg=CFG):
    per = cfg["seq_len"] * cfg["micro_bs"] * cfg["accum"]; steps = int(hours * 3600 * tok_s / per)
    print(f"at {tok_s/1e3:.1f}k tok/s, {hours} GPU-hours = {steps} steps = {steps*per/1e9:.2f}B tokens  -> set CFG['total_steps'] = {steps}")


In [ ]:
# ================= CELL 6a — SMOKE TEST (12 optimizer steps, ~2 min): data -> step -> validation -> throughput =================
smoke = train({**CFG, "run": "smoke"}, steps=12, accum=2, do_eval=True, save=False, resume=False)
rows = [json.loads(l) for l in open(OUT / "smoke" / "log.jsonl")]; tps = [r["tok_s"] for r in rows if "tok_s" in r]
tok_s = float(np.median(tps)) if tps else 15000.0; print(f"\nmeasured throughput: {tok_s/1e3:.1f}k tokens/s")
for h in (4, 6, 8, 10): plan_budget(tok_s, h)
del smoke; gc.collect(); torch.cuda.empty_cache()
print("\nPick the number of GPU-hours you can spend, set CFG['total_steps'] to the printed value, then run cell 6b.")


In [ ]:
# ================= CELL 6b — MAIN RUN: continue the 3,000-step GPT-2 model with the fixed recipe =================
# Edit CFG['total_steps'] first (from cell 6a). The run resumes automatically if the session is interrupted (checkpoint every 20 min).
# GATE: by the 2nd validation line the overall loss must be clearly below the 3,000-step baseline and still falling. If not, stop and report.
CFG["run"] = "gpt2_warm_v1"; CFG["arch"] = "gpt2_warm"
model = train(CFG, minutes_budget=None)      # set minutes_budget=<n> to stop (and save) after n minutes


In [ ]:
# ================= CELL 6c — OPTIONAL architecture pilot (about 10 min): same data/recipe, three same-size models =================
# This measures SPEED and very early learning only. A from-scratch model cannot be fairly ranked against a warm start after ~2M tokens.
for arch in ("gpt2_warm", "gpt2_scratch", "llama_scratch"):
    c = {**CFG, "arch": arch, "run": f"pilot_{arch}", "warmup": 8, "total_steps": 40, "eval_every": 40, "accum": 4}
    train(c, steps=40, do_eval=True, save=False, resume=False)


In [ ]:
# ================= CELL 7 — export: which file to use downstream, plus the registry line for the Phase 3 notebook =================
for run in sorted(p.name for p in OUT.iterdir() if p.is_dir() and (p / "final_model.pt").exists()):
    ck = torch.load(OUT / run / "final_model.pt", map_location="cpu", weights_only=False, mmap=True); mc = ck["model_config"]
    print(f"{run}: {OUT/run/'final_model.pt'}  ({os.path.getsize(OUT/run/'final_model.pt')/2**20:.0f} MB) | step {ck['step']} | config {mc}")
    print(f'   Phase 3 notebook (cell 7) ->  ARCHS["{run}"] = dict(path="{OUT/run/"final_model.pt"}", family="{mc["family"]}")')
    if (OUT / run / "log.jsonl").exists():
        vals = [json.loads(l) for l in open(OUT / run / "log.jsonl") if '"val"' in l]
        if vals: print("   validation history (overall nats/token):", [(v["step"], round(v["val"]["overall"], 3)) for v in vals])
    del ck
print("\nThe GPT-2 family uses the same key layout (transformer.*) as your original checkpoint, so the Phase 3 loader already handles it.")
print("A llama-family checkpoint needs a loader added to the Phase 3 notebook first (send me the file and I will add it).")
